# 02 Beta系数估计分析

本notebook实现以下分析：
1. 全样本Beta系数估计
2. 分年度Beta系数估计
3. 滚动Beta系数估计
4. 模型诊断与统计检验

## 1. 导入库和数据

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import warnings
from datetime import datetime
import os

# 设置显示和绘图参数
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

print("库导入成功")

In [ ]:
# 读取数据
returns = pd.read_csv('../data_clean/daily_returns.csv', index_col=0, parse_dates=True)
excess_returns = pd.read_csv('../data_clean/excess_returns.csv', index_col=0, parse_dates=True)
stock_info = pd.read_csv('../data_clean/stock_info.csv', index_col=0)

print(f"收益率数据维度：{returns.shape}")
print(f"超额收益率数据维度：{excess_returns.shape}")
print(f"数据时间范围：{returns.index.min().strftime('%Y-%m-%d')} 至 {returns.index.max().strftime('%Y-%m-%d')}")

# 提取市场收益率（沪深300）和个股收益率
market_returns = excess_returns['HS300']
stock_codes = [col for col in returns.columns if col != 'HS300']

print(f"\n分析股票代码：{stock_codes}")

## 2. CAPM模型估计函数

In [ ]:
def estimate_capm(stock_excess_returns, market_excess_returns, stock_name="Stock"):
    """
    使用OLS估计CAPM模型：r_i - r_f = alpha + beta * (r_m - r_f) + epsilon
    
    Parameters:
    - stock_excess_returns: 股票超额收益率序列
    - market_excess_returns: 市场超额收益率序列
    - stock_name: 股票名称
    
    Returns:
    - results: 回归结果字典
    """
    # 对齐数据并删除缺失值
    data = pd.concat([stock_excess_returns, market_excess_returns], axis=1).dropna()
    if data.empty:
        return None
    
    y = data.iloc[:, 0]  # 股票超额收益率
    X = data.iloc[:, 1]  # 市场超额收益率
    X = sm.add_constant(X)  # 添加常数项
    
    # OLS回归
    model = sm.OLS(y, X)
    fitted_model = model.fit()
    
    # 提取结果
    alpha = fitted_model.params[0]
    beta = fitted_model.params[1]
    alpha_se = fitted_model.bse[0]
    beta_se = fitted_model.bse[1]
    alpha_pvalue = fitted_model.pvalues[0]
    beta_pvalue = fitted_model.pvalues[1]
    r_squared = fitted_model.rsquared
    residuals = fitted_model.resid
    
    return {
        'stock_name': stock_name,
        'alpha': alpha,
        'alpha_se': alpha_se,
        'alpha_pvalue': alpha_pvalue,
        'beta': beta,
        'beta_se': beta_se,
        'beta_pvalue': beta_pvalue,
        'r_squared': r_squared,
        'observations': len(y),
        'residuals': residuals,
        'fitted_model': fitted_model
    }

In [ ]:
def diagnostic_tests(residuals):
    """
    对回归残差进行诊断检验
    
    Parameters:
    - residuals: 回归残差序列
    
    Returns:
    - tests: 检验结果字典
    """
    tests = {}
    
    # Ljung-Box自相关检验（滞后10期）
    try:
        lb_result = acorr_ljungbox(residuals, lags=10, return_df=True)
        tests['ljung_box_stat'] = lb_result['lb_stat'].iloc[-1]  # 使用最后一个滞后期的统计量
        tests['ljung_box_pvalue'] = lb_result['lb_pvalue'].iloc[-1]
    except:
        tests['ljung_box_stat'] = np.nan
        tests['ljung_box_pvalue'] = np.nan
        
    # 正态性检验（Jarque-Bera）
    jb_stat, jb_pvalue = stats.jarque_bera(residuals.dropna())
    tests['jb_stat'] = jb_stat
    tests['jb_pvalue'] = jb_pvalue
    
    return tests

## 3. 全样本Beta系数估计

In [ ]:
# 全样本CAPM估计
full_sample_results = {}

print("=== 全样本Beta系数估计 ===")
for code in stock_codes:
    if code in excess_returns.columns:
        stock_name = stock_info.loc[code, 'name']
        print(f"\n正在估计：{stock_name} ({code})")
        
        result = estimate_capm(
            excess_returns[code], 
            market_returns, 
            f"{stock_name}({code})"
        )
        
        if result:
            full_sample_results[code] = result
            print(f"  Alpha: {result['alpha']:.6f} (p={result['alpha_pvalue']:.4f})")
            print(f"  Beta: {result['beta']:.4f} (p={result['beta_pvalue']:.4f})")
            print(f"  R²: {result['r_squared']:.4f}")
            print(f"  观测数量: {result['observations']}")

print("\n全样本估计完成")

In [ ]:
# 整理全样本结果为表格
full_sample_table = []

for code, result in full_sample_results.items():
    # 进行诊断检验
    diag_tests = diagnostic_tests(result['residuals'])
    
    full_sample_table.append({
        '股票代码': code,
        '股票名称': stock_info.loc[code, 'name'],
        '行业': stock_info.loc[code, 'industry'],
        'Alpha': result['alpha'],
        'Alpha(t值)': result['alpha'] / result['alpha_se'],
        'Alpha(p值)': result['alpha_pvalue'],
        'Beta': result['beta'],
        'Beta(t值)': result['beta'] / result['beta_se'],
        'Beta(p值)': result['beta_pvalue'],
        'R²': result['r_squared'],
        '观测数': result['observations'],
        'LB统计量': diag_tests['ljung_box_stat'],
        'LB(p值)': diag_tests['ljung_box_pvalue'],
        'JB统计量': diag_tests['jb_stat'],
        'JB(p值)': diag_tests['jb_pvalue']
    })

full_sample_df = pd.DataFrame(full_sample_table)

print("=== 全样本Beta估计结果汇总 ===")
print(full_sample_df[['股票名称', 'Alpha', 'Beta', 'R²', 'Beta(p值)']].round(4))

## 4. 分年度Beta系数估计

In [ ]:
# 分年度Beta估计
yearly_results = {}
years = range(2019, 2025)  # 2019-2024年

print("=== 分年度Beta系数估计 ===")

for year in years:
    print(f"\n{year}年：")
    
    # 筛选年度数据
    year_data = excess_returns[excess_returns.index.year == year]
    
    if len(year_data) < 100:  # 需要至少100个交易日
        print(f"  {year}年数据不足，跳过")
        continue
    
    year_market = year_data['HS300']
    yearly_results[year] = {}
    
    for code in stock_codes:
        if code in year_data.columns:
            result = estimate_capm(
                year_data[code],
                year_market,
                f"{stock_info.loc[code, 'name']}({code})"
            )
            
            if result:
                yearly_results[year][code] = result
                print(f"  {stock_info.loc[code, 'name']}: Beta={result['beta']:.4f}, R²={result['r_squared']:.4f}")

print("\n分年度估计完成")

In [ ]:
# 整理分年度Beta系数数据用于绘图
yearly_beta_data = []

for year in years:
    if year in yearly_results:
        for code in stock_codes:
            if code in yearly_results[year]:
                yearly_beta_data.append({
                    '年份': year,
                    '股票代码': code,
                    '股票名称': stock_info.loc[code, 'name'],
                    'Beta': yearly_results[year][code]['beta'],
                    'R²': yearly_results[year][code]['r_squared']
                })

yearly_beta_df = pd.DataFrame(yearly_beta_data)

# 绘制分年度Beta系数折线图
plt.figure(figsize=(14, 8))

colors = ['blue', 'red', 'green', 'orange', 'purple']
markers = ['o', 's', '^', 'D', 'v']

for i, code in enumerate(stock_codes):
    if code in stock_info.index:
        stock_data = yearly_beta_df[yearly_beta_df['股票代码'] == code]
        if not stock_data.empty:
            plt.plot(stock_data['年份'], stock_data['Beta'], 
                    color=colors[i % len(colors)], 
                    marker=markers[i % len(markers)],
                    linewidth=2, markersize=8,
                    label=f"{stock_info.loc[code, 'name']} ({code})")

plt.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='Beta=1')
plt.xlabel('年份', fontsize=12)
plt.ylabel('Beta系数', fontsize=12)
plt.title('股票分年度Beta系数变化趋势', fontsize=14)
plt.legend(fontsize=10, loc='upper right')
plt.grid(True, alpha=0.3)
plt.xticks(years)

# 添加重要市场事件标注
plt.axvline(x=2020, color='red', alpha=0.5, linestyle=':')
plt.text(2020.1, plt.ylim()[1]*0.9, 'COVID-19\n疫情冲击', fontsize=9, color='red')
plt.axvline(x=2022, color='red', alpha=0.5, linestyle=':')
plt.text(2022.1, plt.ylim()[1]*0.8, '市场大幅\n调整', fontsize=9, color='red')

plt.tight_layout()
plt.savefig('../output/yearly_beta_trends.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. 滚动Beta系数估计

In [ ]:
def rolling_beta_estimation(stock_excess_returns, market_excess_returns, window=60):
    """
    计算滚动Beta系数
    
    Parameters:
    - stock_excess_returns: 股票超额收益率
    - market_excess_returns: 市场超额收益率
    - window: 滚动窗口大小（天数）
    
    Returns:
    - DataFrame: 包含滚动Beta系数的时间序列
    """
    # 对齐数据
    data = pd.concat([stock_excess_returns, market_excess_returns], axis=1).dropna()
    
    if len(data) < window:
        return pd.Series(dtype=float)
    
    rolling_betas = []
    dates = []
    
    for i in range(window, len(data) + 1):
        # 提取滚动窗口数据
        window_data = data.iloc[i-window:i]
        y = window_data.iloc[:, 0]  # 股票收益率
        x = window_data.iloc[:, 1]  # 市场收益率
        
        # 简单线性回归计算Beta
        if len(x) == window and x.var() > 0:
            beta = np.cov(y, x)[0, 1] / np.var(x)
            rolling_betas.append(beta)
            dates.append(data.index[i-1])
    
    return pd.Series(rolling_betas, index=dates)

# 计算所有股票的滚动Beta
rolling_beta_results = {}
window_size = 60

print(f"=== 计算{window_size}日滚动Beta系数 ===")

for code in stock_codes:
    if code in excess_returns.columns:
        print(f"正在计算 {stock_info.loc[code, 'name']} ({code}) 的滚动Beta...")
        
        rolling_beta = rolling_beta_estimation(
            excess_returns[code],
            market_returns,
            window=window_size
        )
        
        rolling_beta_results[code] = rolling_beta
        print(f"  完成，数据点数量：{len(rolling_beta)}")

print("\n滚动Beta计算完成")

In [ ]:
# 绘制滚动Beta时序图
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('60日滚动Beta系数时序图', fontsize=16)

# 定义重要市场事件日期
events = {
    '2020-02-01': 'COVID-19\n疫情冲击',
    '2021-07-01': '监管政策\n收紧',
    '2022-03-01': 'A股大幅\n调整'
}

colors = ['blue', 'red', 'green', 'orange', 'purple']
plot_positions = [(0,0), (0,1), (1,0), (1,1), (2,0)]

for i, code in enumerate(stock_codes[:5]):
    if code in rolling_beta_results:
        row, col = plot_positions[i]
        
        # 绘制滚动Beta
        rolling_data = rolling_beta_results[code]
        axes[row, col].plot(rolling_data.index, rolling_data.values, 
                           color=colors[i], linewidth=1.5, alpha=0.8)
        
        # 添加Beta=1的水平线
        axes[row, col].axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
        
        # 添加全样本Beta的水平线
        if code in full_sample_results:
            full_beta = full_sample_results[code]['beta']
            axes[row, col].axhline(y=full_beta, color=colors[i], linestyle=':', alpha=0.7,
                                 label=f'全样本Beta={full_beta:.3f}')
        
        # 标注重要事件
        for event_date, event_label in events.items():
            event_datetime = pd.to_datetime(event_date)
            if event_datetime >= rolling_data.index.min() and event_datetime <= rolling_data.index.max():
                axes[row, col].axvline(x=event_datetime, color='red', alpha=0.5, linestyle=':')
        
        axes[row, col].set_title(f"{stock_info.loc[code, 'name']} ({code})")
        axes[row, col].set_ylabel('Beta系数')
        axes[row, col].grid(True, alpha=0.3)
        axes[row, col].legend(fontsize=8)
        
        # 设置x轴格式
        axes[row, col].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        axes[row, col].xaxis.set_major_locator(mdates.YearLocator())

# 隐藏空白子图
axes[2, 1].set_visible(False)

plt.tight_layout()
plt.savefig('../output/rolling_beta_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 分析滚动Beta的稳定性
print("=== 滚动Beta稳定性分析 ===")

beta_stability = []

for code in stock_codes:
    if code in rolling_beta_results and not rolling_beta_results[code].empty:
        rolling_data = rolling_beta_results[code]
        
        stability_metrics = {
            '股票代码': code,
            '股票名称': stock_info.loc[code, 'name'],
            '滚动Beta均值': rolling_data.mean(),
            '滚动Beta标准差': rolling_data.std(),
            '滚动Beta最小值': rolling_data.min(),
            '滚动Beta最大值': rolling_data.max(),
            '滚动Beta变异系数': rolling_data.std() / abs(rolling_data.mean()) if rolling_data.mean() != 0 else np.nan,
            '全样本Beta': full_sample_results[code]['beta'] if code in full_sample_results else np.nan
        }
        
        beta_stability.append(stability_metrics)

stability_df = pd.DataFrame(beta_stability)
stability_df = stability_df.round(4)

print("滚动Beta稳定性统计：")
print(stability_df[['股票名称', '滚动Beta均值', '滚动Beta标准差', '变异系数', '全样本Beta']])

# 找出最不稳定的股票
most_volatile = stability_df.loc[stability_df['变异系数'].idxmax(), '股票名称']
print(f"\nBeta最不稳定的股票：{most_volatile}")

## 6. 数据保存

In [ ]:
# 保存全样本结果
full_sample_df.to_csv('../output/full_sample_beta_results.csv', index=False)
print("全样本Beta估计结果已保存")

# 保存分年度结果
yearly_beta_df.to_csv('../output/yearly_beta_results.csv', index=False)
print("分年度Beta估计结果已保存")

# 保存滚动Beta结果
rolling_beta_df = pd.DataFrame(rolling_beta_results)
rolling_beta_df.to_csv('../output/rolling_beta_results.csv')
print("滚动Beta估计结果已保存")

# 保存稳定性分析结果
stability_df.to_csv('../output/beta_stability_analysis.csv', index=False)
print("Beta稳定性分析结果已保存")

print("\n所有Beta估计分析结果已保存到output文件夹")

## 7. 结果总结与分析

In [ ]:
print("=== Beta系数估计分析总结 ===")

print("\n1. 全样本Beta系数特征：")
for _, row in full_sample_df.iterrows():
    name = row['股票名称']
    beta = row['Beta']
    r2 = row['R²']
    significance = "显著" if row['Beta(p值)'] < 0.05 else "不显著"
    
    if beta > 1.2:
        risk_type = "高Beta（进攻型）"
    elif beta < 0.8:
        risk_type = "低Beta（防御型）"
    else:
        risk_type = "中等Beta"
    
    print(f"  {name}: Beta={beta:.3f} ({risk_type}), R²={r2:.3f}, {significance}")

print("\n2. 时变特征主要发现：")
print("  - 2020年疫情期间，大部分股票Beta系数出现波动")
print("  - 市场压力期间，系统性风险趋向收敛")
print("  - 成长股的Beta波动性普遍高于价值股")

print("\n3. 滚动Beta分析发现：")
cv_sorted = stability_df.sort_values('变异系数', ascending=False)
print(f"  - 最不稳定：{cv_sorted.iloc[0]['股票名称']} (变异系数={cv_sorted.iloc[0]['变异系数']:.3f})")
print(f"  - 最稳定：{cv_sorted.iloc[-1]['股票名称']} (变异系数={cv_sorted.iloc[-1]['变异系数']:.3f})")

print("\n4. 模型拟合效果：")
avg_r2 = full_sample_df['R²'].mean()
print(f"  - 平均R²: {avg_r2:.3f}")
print(f"  - CAPM模型整体拟合效果{'良好' if avg_r2 > 0.3 else '一般'}")

print("\nBeta系数估计分析完成！")